# GRPO

使用组内相对奖励代替 Critic 的 Group Relative Policy Optimization。

## 核心思想

GRPO（Group Relative Policy Optimization）最初用于大语言模型的强化学习：对同一个问题采样一组回答，再用回答之间的相对奖励估计优势。它保留 PPO 的概率比率截断，但不再训练价值网络。

为了在经典控制环境中展示算法，本 notebook 将连续采样的 `group_size` 条 CartPole 轨迹视为一组，并以每条轨迹的总回报作为结果奖励。组内标准化后的奖励会分配给该轨迹的所有动作。这个改写保留了 GRPO 的关键机制，但不是语言模型训练系统的完整复现。

## 组相对优势

一组轨迹的结果奖励为 $R_1,\ldots,R_G$。GRPO 不使用 Critic，而是用组内统计量构造优势：

$$
\hat A_i=\frac{R_i-\operatorname{mean}(R_1,\ldots,R_G)}{\operatorname{std}(R_1,\ldots,R_G)+\varepsilon}.
$$

本例把同一条轨迹的 $\hat A_i$ 分配给轨迹中的每个时间步。这样，高于组内平均水平的轨迹会被提高概率，低于平均水平的轨迹会被降低概率。

## 截断目标与 KL 约束

和 PPO 一样，先用旧策略 $\pi_{\theta_{\mathrm{old}}}$ 收集数据，再计算概率比率

$$
r_t(\theta)=\frac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{\mathrm{old}}}(a_t\mid s_t)}.
$$

单个样本的优化目标为

$$
\min\left(r_t\hat A_i,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)\hat A_i\right)-\beta D_{\mathrm{KL}}(\pi_\theta\|\pi_{\mathrm{ref}}).
$$

其中参考策略 `ref_policy` 在训练开始时冻结，KL 项抑制当前策略偏离参考策略过远。代码使用只依赖采样动作的非负 KL 估计

$$
\exp(x)-x-1,\qquad x=\log\pi_{\mathrm{ref}}(a\mid s)-\log\pi_\theta(a\mid s).
$$

## 代码实现

### 导入依赖与环境兼容函数

In [ ]:
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

try:
    import gymnasium as gym
except ImportError:
    import gym


def reset_env(env, seed=None):
    result = env.reset(seed=seed)
    return result[0] if isinstance(result, tuple) else result

def step_env(env, action):
    result = env.step(action)
    if len(result) == 5:
        next_state, reward, terminated, truncated, info = result
        return next_state, reward, terminated or truncated, info
    return result

### 策略网络

策略网络输出离散动作概率。GRPO 不需要价值网络。

In [ ]:
class PolicyNet(torch.nn.Module):
    def __init__(self, state_dim, hidden_dim, action_dim):
        super().__init__()
        self.fc1 = torch.nn.Linear(state_dim, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return F.softmax(self.fc2(x), dim=-1)

### GRPO 智能体

`collect_trajectory` 用当前策略采样完整轨迹并保存旧策略的动作对数概率。`update` 先计算组内相对优势，再复用整组数据训练多个 epoch。

In [ ]:
class GRPO:
    def __init__(self, state_dim, hidden_dim, action_dim, actor_lr,
                 epochs, eps, beta, device):
        self.actor = PolicyNet(state_dim, hidden_dim, action_dim).to(device)
        self.ref_policy = copy.deepcopy(self.actor).to(device).eval()
        for parameter in self.ref_policy.parameters():
            parameter.requires_grad_(False)
        self.optimizer = torch.optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.epochs = epochs
        self.eps = eps
        self.beta = beta
        self.device = device

    @torch.no_grad()
    def take_action(self, state):
        state = torch.as_tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
        distribution = torch.distributions.Categorical(self.actor(state))
        action = distribution.sample()
        return action.item(), distribution.log_prob(action).item()

    def collect_trajectory(self, env, seed=None):
        states, actions, old_log_probs, rewards = [], [], [], []
        state = reset_env(env, seed)
        done = False
        while not done:
            action, old_log_prob = self.take_action(state)
            next_state, reward, done, _ = step_env(env, action)
            states.append(state)
            actions.append(action)
            old_log_probs.append(old_log_prob)
            rewards.append(reward)
            state = next_state
        return {
            'states': states,
            'actions': actions,
            'old_log_probs': old_log_probs,
            'return': float(sum(rewards)),
        }

    def update(self, trajectories):
        returns = torch.tensor(
            [trajectory['return'] for trajectory in trajectories],
            dtype=torch.float32, device=self.device
        )
        relative_advantages = (returns - returns.mean()) / (returns.std(unbiased=False) + 1e-8)

        states, actions, old_log_probs, advantages = [], [], [], []
        for trajectory, advantage in zip(trajectories, relative_advantages):
            length = len(trajectory['actions'])
            states.extend(trajectory['states'])
            actions.extend(trajectory['actions'])
            old_log_probs.extend(trajectory['old_log_probs'])
            advantages.extend([advantage.item()] * length)

        states = torch.as_tensor(np.asarray(states), dtype=torch.float32, device=self.device)
        actions = torch.as_tensor(actions, dtype=torch.long, device=self.device)
        old_log_probs = torch.as_tensor(old_log_probs, dtype=torch.float32, device=self.device)
        advantages = torch.as_tensor(advantages, dtype=torch.float32, device=self.device)

        with torch.no_grad():
            ref_distribution = torch.distributions.Categorical(self.ref_policy(states))
            ref_log_probs = ref_distribution.log_prob(actions)

        for _ in range(self.epochs):
            distribution = torch.distributions.Categorical(self.actor(states))
            log_probs = distribution.log_prob(actions)
            ratio = torch.exp(log_probs - old_log_probs)
            surrogate_1 = ratio * advantages
            surrogate_2 = torch.clamp(
                ratio, 1 - self.eps, 1 + self.eps
            ) * advantages

            log_ratio_to_ref = ref_log_probs - log_probs
            kl_estimate = torch.exp(log_ratio_to_ref) - log_ratio_to_ref - 1
            objective = torch.min(surrogate_1, surrogate_2) - self.beta * kl_estimate
            loss = -objective.mean()

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 1.0)
            self.optimizer.step()

        return {
            'mean_return': returns.mean().item(),
            'return_std': returns.std(unbiased=False).item(),
            'loss': loss.item(),
            'kl': kl_estimate.mean().item(),
        }

### 训练

一次迭代先完整采样一组轨迹，再统一更新策略。`group_size` 越大，组内基线通常越稳定，但每次更新需要更多环境交互。

In [ ]:
actor_lr = 1e-3
num_iterations = 100
group_size = 8
hidden_dim = 128
epochs = 5
eps = 0.2
beta = 1e-3
seed = 0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

env_name = 'CartPole-v1'
env = gym.make(env_name)
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n
agent = GRPO(state_dim, hidden_dim, action_dim, actor_lr, epochs, eps, beta, device)

episode_returns = []
group_mean_returns = []
for iteration in range(num_iterations):
    trajectories = [
        agent.collect_trajectory(env, seed + iteration * group_size + index)
        for index in range(group_size)
    ]
    metrics = agent.update(trajectories)
    episode_returns.extend(trajectory['return'] for trajectory in trajectories)
    group_mean_returns.append(metrics['mean_return'])

    if (iteration + 1) % 10 == 0:
        print(
            f"Iteration {iteration + 1:3d} | "
            f"mean return {metrics['mean_return']:7.2f} | "
            f"KL {metrics['kl']:.5f}"
        )

env.close()

In [ ]:
def moving_average(values, window_size):
    values = np.asarray(values, dtype=np.float32)
    if len(values) < window_size:
        return values
    kernel = np.ones(window_size, dtype=np.float32) / window_size
    return np.convolve(values, kernel, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(episode_returns, alpha=0.25, label='Episode return')
window_size = 25
smoothed_returns = moving_average(episode_returns, window_size)
plt.plot(
    np.arange(window_size - 1, len(episode_returns)),
    smoothed_returns,
    label=f'{window_size}-episode moving average'
)
plt.xlabel('Episodes')
plt.ylabel('Returns')
plt.title(f'GRPO on {env_name}')
plt.legend()
plt.show()

## 与 PPO 的对比

| | PPO | GRPO |
| --- | --- | --- |
| 优势来源 | Critic + GAE | 组内相对奖励 |
| 是否训练 Critic | 是 | 否 |
| 策略稳定机制 | 概率比率截断 | 概率比率截断 + 参考策略 KL |
| 每批数据 | 一条或多条轨迹 | 必须包含可比较的一组轨迹 |

GRPO 省去了价值网络，但效果依赖组内奖励是否具有可比性。对于语言模型，同一个 prompt 的多个回答天然构成一组；在本例中，CartPole 的任务固定，因此用相邻采样的多条轨迹近似这一结构。